## Marks

What `probatum.chart.marks` draws besides a line: glyphs where a condition holds, regimes
shaded behind the bars, colours that follow a value, a cloud carried past the last candle,
and the price itself drawn another way. Each mark takes your own arrays, one value a bar,
here on daily Bitcoin bars of 2024 and 2025.

In [1]:
import pathlib

import exchange_router
import numpy
import probatum

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}")

probatum 3.0.0, exchange-router 3.3.0


### The bars

Binance's hourly BTCUSDT, fetched once into `bars/` and resampled to days.

In [2]:
PATH = pathlib.Path("../bars/binance-spot-btcusdt-1h-2024-2025")

if PATH.exists():
    hourly = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["binance"]) as router:
        hourly = probatum.Data.from_router(router, exchange="binance", market="spot",
                                           interval="1h", symbols=["BTCUSDT"],
                                           start="2024-01-01", end="2025-12-31")
    hourly.save(path=PATH)

days = hourly.resample(interval="1d")

high, low, close, volume = days.high, days.low, days.close, days.volume

days

Data(BTCUSDT, 1d, 731 bars from 2024-01-01 00:00 to 2025-12-31 00:00)

### Glyphs where a condition holds

`Markers` puts one glyph on every bar a boolean array holds, here where the 21-day average
crosses the 55-day one, and `Marker` one annotation on one bar. `offset` is a distance in
pixels, positive upward, so a glyph keeps its place at every zoom.

In [3]:
fast = probatum.ta.ema(close, length=21)
slow = probatum.ta.ema(close, length=55)
peak = int(numpy.nanargmax(high))

days.chart(
    probatum.chart.marks.Line(fast, name="EMA 21"),
    probatum.chart.marks.Line(slow, name="EMA 55", style="dashed"),
    probatum.chart.marks.Markers(probatum.ta.crossover(fast, other=slow), shape="arrow_up",
                                 offset=-14, color="#089981"),
    probatum.chart.marks.Markers(probatum.ta.crossunder(fast, other=slow),
                                 shape="arrow_down", offset=14, color="#f23645"),
    probatum.chart.marks.Marker(peak, text=f"high {high[peak]:,.0f}", shape="flag",
                                offset=16),
    focus=365,
    title="Where the 21-day average crosses the 55-day, BTCUSDT 1d",
)

Chart(bars=731, panels=2)

### Regimes behind the bars

`Background` shades the bars a label names, each label in a colour of its own, and its
`name` puts the region under the crosshair in the legend. Here the average true range as a
share of the close, against its own mean of the last hundred days.

In [4]:
natr  = probatum.ta.natr(high, low=low, close=close, length=14)
usual = probatum.ta.sma(natr, length=100)

regime = numpy.where(natr > usual * 1.25, "wide",
                     numpy.where(natr < usual * 0.8, "narrow", "usual"))

days.chart(
    probatum.chart.marks.Background(regime, name="range",
                                    fill={"wide": "#f2a33c2e", "narrow": "#8b97a524"}),
    probatum.chart.marks.Line(natr, name="NATR 14, %", panel="range"),
    probatum.chart.marks.Line(usual, name="its 100-day mean", panel="range", width=1,
                              style="dashed"),
    focus=365,
    title="Daily range against its own past, BTCUSDT 1d",
)

Chart(bars=731, panels=3)

### Colours that follow a value

`color` takes one colour or one a bar, and `ramp` turns numbers into colours, so a line or
a histogram carries a second variable in its own colour: here the 30-day rate of change,
pinned to a domain of 20% either way. `candle_color` tints the candles themselves, here on
the days that traded twice their month's mean volume.

In [5]:
change = probatum.ta.roc(close, length=30)
shades = probatum.chart.marks.ramp(change, colors=["#f23645", "#8b97a5", "#089981"],
                                   domain=(-20.0, 20.0))

days.chart(
    probatum.chart.marks.Line(probatum.ta.sma(close, length=50), name="SMA 50", width=3,
                              color=shades),
    probatum.chart.marks.Histogram(change, name="ROC 30, %", panel="change", color=shades),
    candle_color=numpy.where(volume > 2.0 * probatum.ta.sma(volume, length=30), "#4d9fff",
                             None),
    focus=365,
    title="The 50-day average coloured by the month's change, BTCUSDT 1d",
)

Chart(bars=731, panels=3)

### A cloud past the last candle

Ichimoku's cloud is computed from real bars but drawn 26 bars ahead of them. The 26 places
in front are the series' own, and `future=26` gives the axis room past the last candle;
the fill takes one colour a bar, green where the first edge is on top.

In [6]:
def midpoint(length):
    return (probatum.ta.highest(high, length=length)
            + probatum.ta.lowest(low, length=length)) / 2.0


conversion = midpoint(length=9)
base       = midpoint(length=26)

ahead  = numpy.full(shape=26, fill_value=numpy.nan)
lead_a = numpy.concatenate([ahead, (conversion + base) / 2.0])
lead_b = numpy.concatenate([ahead, midpoint(length=52)])

days.chart(
    probatum.chart.marks.Band(lead_a, lower=lead_b, name="cloud",
                              fill=numpy.where(lead_a >= lead_b, "#08998140", "#f2364540")),
    probatum.chart.marks.Line(conversion, name="conversion 9", width=1),
    probatum.chart.marks.Line(base, name="base 26", width=1, style="dashed"),
    future=26,
    focus=180,
    title="Ichimoku's cloud, 26 days ahead, BTCUSDT 1d",
)

Chart(bars=731, panels=2)

### The price another way

`price=` draws hollow candles, whose rising bodies are outlines, bars, or a line of the
closes, here over the last four months. `show()` draws a chart anywhere in a cell.

In [7]:
recent = days.after(when="2025-09-01")

for form in ["hollow", "bars", "line"]:
    recent.chart(price=form, title=f'price="{form}", BTCUSDT 1d', height=360).show()